# Notebook 12 -- Build centroid_chars.csv and firm_panel.csv

Constructs the two firm-level output files required by notebooks 05-09
from the cluster panel and the imputed characteristics file.

**Input files:**
- `Results/cluster_month_panels_K_50/cluster_month_panel_K_50_lambda_1000000.csv`
- `Data/Imputed__characteristics_winsorized99.csv` (or the inner-join version)

**Output files** (written to `replication_repo/data/`):
- `centroid_chars.csv` — VW mean of 45 characteristics per (year_month, cluster)
- `firm_panel.csv`     — firm-level panel with cluster, ret, me, SIC, 45 chars

**Run cells in order.** Cell 6 detects whether the characteristics file
already has a `cluster` column. If it does, Cells 7-8 produce both output
files directly. If not, Cell 9 explains the fallback options.

In [1]:
import os, sys
import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO_ROOT)

# ── Paths ──────────────────────────────────────────────────────────
DATA_PATH    = "/ssd1/songjiangliu/shared/asset_clustering/Data"
RESULTS_PATH = "/ssd1/songjiangliu/shared/asset_clustering/Results"
PANEL_FILE   = os.path.join(RESULTS_PATH,
               "cluster_month_panels_K_50",
               "cluster_month_panel_K_50_lambda_1000000.csv")

# Output goes to replication_repo/data/
OUT_CENTROID = os.path.join(REPO_ROOT, "data", "centroid_chars.csv")
OUT_FIRM     = os.path.join(REPO_ROOT, "data", "firm_panel.csv")

print("Panel file :", PANEL_FILE)
print("Chars file : (see Cell 2)")
print("Output     :", OUT_CENTROID)

Panel file : /ssd1/songjiangliu/shared/asset_clustering/Results/cluster_month_panels_K_50/cluster_month_panel_K_50_lambda_1000000.csv
Chars file : (see Cell 2)
Output     : /ssd1/songjiangliu/shared/asset_clustering/Replication Package/data/centroid_chars.csv


In [2]:
# ── Load cluster-month panel (cluster assignments + returns) ───────
print("Loading cluster panel...")
panel = pd.read_csv(PANEL_FILE)
panel = panel[["year_month", "cluster"]].drop_duplicates()
print(f"  Panel: {panel.shape}  |  {panel['year_month'].nunique()} months  |  {panel['cluster'].nunique()} clusters")
print(f"  Period: {panel['year_month'].min()}  ->  {panel['year_month'].max()}")
panel.head(3)

Loading cluster panel...
  Panel: (26395, 2)  |  528 months  |  50 clusters
  Period: 1977-01  ->  2020-12


,year_month,cluster
0,1977-01,0
1,1977-02,0
2,1977-03,0


In [3]:
# ── Load firm-level characteristics ───────────────────────────────
# Use the winsorized file (smaller, 260 MB) if it has PERMNO + year_month.
# Fall back to the inner-join file which also has MktCap and SICCD.
#
# Try winsorized first; if it lacks PERMNO/date columns use the larger file.

CHARS_FILE = os.path.join(DATA_PATH, "Imputed__characteristics_winsorized99.csv")
CHARS_FILE_FULL = os.path.join(DATA_PATH,
                  "Imputed__characteristics_with_MktCap_SICCD_(inner_join).csv")

print(f"Loading characteristics... (this may take 1-2 minutes)")

# Read just the header first to check columns
header = pd.read_csv(CHARS_FILE, nrows=0)
print(f"  Winsorized file columns ({len(header.columns)}): {list(header.columns[:10])}...")

# Decide which columns identify firm + date
# Common patterns: permno/PERMNO, year_month, date, DATE, yyyymm
id_candidates = [c for c in header.columns
                 if c.lower() in ("permno","year_month","date","yyyymm","mktcap",
                                  "siccd","sic","year","month")]
print(f"  ID-like columns found: {id_candidates}")

Loading characteristics... (this may take 1-2 minutes)
  Winsorized file columns (50): ['date', 'permno', 'A2ME', 'AC', 'AT', 'ATO', 'B2M', 'BETA_d', 'BETA_m', 'C2A']...
  ID-like columns found: ['date', 'permno', 'year_month']


In [4]:
# ── Read the characteristics file with correct ID columns ──────────
# Adjust these if the print above shows different column names

PERMNO_COL    = "permno"       # firm identifier — change if different
DATE_COL      = "year_month"   # date column     — change if different

# Load (use the full file if winsorized lacks PERMNO or date)
chars = pd.read_csv(CHARS_FILE, low_memory=False)
print(f"Chars loaded: {chars.shape}")
print(f"Columns: {list(chars.columns)}")

# Normalise column names to lowercase
chars.columns = [c.strip().lower() for c in chars.columns]
PERMNO_COL = PERMNO_COL.lower()
DATE_COL   = DATE_COL.lower()

# Check they exist
assert PERMNO_COL in chars.columns, f"'{PERMNO_COL}' not found. Cols: {list(chars.columns)}"
assert DATE_COL   in chars.columns, f"'{DATE_COL}' not found. Cols: {list(chars.columns)}"

print(f"\nFirm column : {PERMNO_COL}")
print(f"Date column : {DATE_COL}")
print(f"Date range  : {chars[DATE_COL].min()}  ->  {chars[DATE_COL].max()}")
print(f"Unique firms: {chars[PERMNO_COL].nunique():,}")
print(f"Unique dates: {chars[DATE_COL].nunique():,}")

Chars loaded: (286000, 50)
Columns: ['date', 'permno', 'A2ME', 'AC', 'AT', 'ATO', 'B2M', 'BETA_d', 'BETA_m', 'C2A', 'CF2B', 'CF2P', 'CTO', 'D2A', 'D2P', 'DPI2A', 'E2P', 'FC2Y', 'HIGH52', 'INV', 'IdioVol', 'LEV', 'ME', 'NI', 'NOA', 'OA', 'OL', 'OP', 'PCM', 'PM', 'PROF', 'Q', 'R12_2', 'R12_7', 'R2_1', 'R36_13', 'R60_13', 'RNA', 'ROA', 'ROE', 'RVAR', 'S2P', 'SGA2S', 'SPREAD', 'SUV', 'TURN', 'VAR', 'return', 'monthly_update', 'year_month']

Firm column : permno
Date column : year_month
Date range  : 1977-01  ->  1982-10
Unique firms: 5,448
Unique dates: 70


In [5]:
# ── Identify the 45 characteristic columns ─────────────────────────
# These are all numeric columns that are NOT identifier columns
NON_CHAR_COLS = {PERMNO_COL, DATE_COL, "mktcap", "mktcap_sum",
                 "siccd", "sic", "year", "month", "date",
                 "year_month", "permno", "retadj", "ret", "prc",
                 "shrout", "exchcd", "shrcd", "dlret", "dlstcd"}

CHARS_45 = [
    "a2me","ac","at","ato","b2m","beta_d","beta_m","c2a","cf2b","cf2p",
    "cto","d2a","d2p","dpi2a","e2p","fc2y","high52","idiovol","inv","lev",
    "me","ni","noa","oa","ol","op","pcm","pm","prof","q",
    "r12_2","r12_7","r2_1","r36_13","r60_13","rna","roa","roe","rvar",
    "s2p","sga2s","spread","suv","turn","var"
]

# Find which of the 45 are actually present in the file
present = [c for c in CHARS_45 if c in chars.columns]
missing = [c for c in CHARS_45 if c not in chars.columns]

print(f"Expected 45 characteristics")
print(f"  Found  : {len(present)}")
print(f"  Missing: {len(missing)}")
if missing:
    print(f"  Missing cols: {missing}")

# If none found, print all numeric cols so we can identify them
if len(present) == 0:
    num_cols = chars.select_dtypes(include=[np.number]).columns.tolist()
    id_cols  = [c for c in num_cols if c in NON_CHAR_COLS]
    char_cols_all = [c for c in num_cols if c not in NON_CHAR_COLS]
    print(f"\nAll numeric non-ID columns ({len(char_cols_all)}): {char_cols_all}")
    USE_COLS = char_cols_all
else:
    USE_COLS = present

print(f"\nUsing {len(USE_COLS)} characteristic columns: {USE_COLS[:8]}...")

Expected 45 characteristics
  Found  : 45
  Missing: 0

Using 45 characteristic columns: ['a2me', 'ac', 'at', 'ato', 'b2m', 'beta_d', 'beta_m', 'c2a']...


In [10]:
import os

base = "/ssd1/songjiangliu/shared/asset_clustering"

for root, dirs, files in os.walk(base):
    # Skip heavy data dirs to keep it fast
    dirs[:] = [d for d in dirs if d not in 
               ("Data", ".git", "__pycache__", "node_modules")]
    for f in files:
        if f.endswith((".csv", ".parquet", ".pkl", ".feather", ".h5")):
            full = os.path.join(root, f)
            size = os.path.getsize(full) / 1e6
            if size > 5:   # files larger than 5MB only
                print(f"{size:8.1f} MB  {full}")

    24.5 MB  /ssd1/songjiangliu/shared/asset_clustering/cluster_centers.csv
   221.0 MB  /ssd1/songjiangliu/shared/asset_clustering/Results/cluster_characteristic_averages_all50.csv
    17.6 MB  /ssd1/songjiangliu/shared/asset_clustering/Results/cluster_characteristic_averages.csv
   171.1 MB  /ssd1/songjiangliu/shared/asset_clustering/Results/inertia/clustering_results_K_50.csv
   168.9 MB  /ssd1/songjiangliu/shared/asset_clustering/Results/inertia/clustering_results_K_10.csv
  2694.3 MB  /ssd1/songjiangliu/shared/asset_clustering/Results/clustering_results_with_characteristics/characteristics_clustering_results_K_50_lambda_1000.csv
  2663.3 MB  /ssd1/songjiangliu/shared/asset_clustering/Results/clustering_results_with_characteristics/characteristics_clustering_results_K_50.csv
  2694.1 MB  /ssd1/songjiangliu/shared/asset_clustering/Results/clustering_results_with_characteristics/characteristics_clustering_results_K_50_lambda_1000000.csv
  2663.3 MB  /ssd1/songjiangliu/shared/asset_cl

In [6]:
# ── Merge cluster assignments into firm-level panel ─────────────────
# Panel has year_month + cluster (one row per cluster per month)
# Chars has permno + year_month + characteristics (one row per firm per month)
# We need to assign each firm to a cluster for each month.
#
# The cluster panel does NOT have firm-level assignments — it only has
# the aggregate (cluster, month) level. We need a firm->cluster mapping.
#
# Check if the full characteristics file has a cluster column already

if "cluster" in chars.columns:
    print("'cluster' column already present in characteristics file — using it directly.")
    firm_chars = chars.copy()
else:
    print("No 'cluster' column in characteristics file.")
    print("We need the firm-level cluster assignment file.")
    print()
    print("Looking for cluster assignment files in Results/...")
    
    # Sometimes there is a firm_cluster_assignments file
    for root, dirs, files in os.walk(RESULTS_PATH):
        for f in files:
            if "firm" in f.lower() or "assign" in f.lower() or "permno" in f.lower():
                print(f"  Found: {os.path.join(root, f)}")
    
    print()
    print("If no assignment file found, the cluster assignments must come from")
    print("your clustering code output. Set firm_chars = None and see Cell 7.")

No 'cluster' column in characteristics file.
We need the firm-level cluster assignment file.

Looking for cluster assignment files in Results/...
  Found: /ssd1/songjiangliu/shared/asset_clustering/Results/famamacbeth_permno_no_agg.xlsx
  Found: /ssd1/songjiangliu/shared/asset_clustering/Results/temporal_loss/plots/hl_return_lines_vs_K_lagged_assignments.png

If no assignment file found, the cluster assignments must come from
your clustering code output. Set firm_chars = None and see Cell 7.


In [ ]:
# ── If cluster column is present: build centroid_chars.csv ─────────
# This cell runs when chars already has a 'cluster' column.

if "cluster" not in chars.columns:
    print("Skipping — no cluster column. See Cell 6 output for next steps.")
else:
    firm_chars = chars.rename(columns={DATE_COL: "year_month",
                                        PERMNO_COL: "permno"})
    
    # ── centroid_chars.csv ──────────────────────────────────────────
    # Value-weighted mean of each characteristic per (year_month, cluster)
    # Weight = mktcap if available, else equal-weight
    
    if "mktcap" in firm_chars.columns:
        print("Computing value-weighted centroids (weight = mktcap)...")
        def vw_mean(g):
            w = g["mktcap"].fillna(0)
            if w.sum() == 0:
                w = pd.Series(1.0, index=g.index)
            out = {}
            for c in USE_COLS:
                vals = g[c]
                mask = vals.notna() & w.notna()
                out[c] = (vals[mask] * w[mask]).sum() / w[mask].sum() if mask.sum() > 0 else np.nan
            return pd.Series(out)
        
        centroid = (firm_chars
                    .groupby(["year_month", "cluster"])
                    .apply(vw_mean)
                    .reset_index())
    else:
        print("Computing equal-weighted centroids (no mktcap column found)...")
        centroid = (firm_chars
                    .groupby(["year_month", "cluster"])[USE_COLS]
                    .mean()
                    .reset_index())
    
    print(f"Centroid chars shape: {centroid.shape}")
    print(f"Sample:")
    print(centroid.head(3).to_string())
    
    centroid.to_csv(OUT_CENTROID, index=False)
    print(f"\nSaved -> {OUT_CENTROID}")

In [ ]:
# ── Build firm_panel.csv ────────────────────────────────────────────
# Required columns: year_month, permno, cluster, ret, me, SIC, [45 chars]

if "cluster" not in chars.columns:
    print("Skipping — no cluster column.")
else:
    keep_cols = ["year_month", "permno", "cluster"] + USE_COLS
    
    # Add ret (return), me (market equity), SIC if present
    for extra in ["ret","retadj","me","mktcap","siccd","sic"]:
        if extra in firm_chars.columns and extra not in keep_cols:
            keep_cols.append(extra)
    
    # Rename to standard names expected by notebooks
    rename_map = {}
    if "retadj" in keep_cols and "ret" not in firm_chars.columns:
        rename_map["retadj"] = "ret"
    if "mktcap" in keep_cols and "me" not in firm_chars.columns:
        rename_map["mktcap"] = "me"
    if "siccd" in keep_cols and "SIC" not in firm_chars.columns:
        rename_map["siccd"] = "SIC"
    
    firm_panel = firm_chars[[c for c in keep_cols if c in firm_chars.columns]].copy()
    firm_panel.rename(columns=rename_map, inplace=True)
    
    print(f"Firm panel shape : {firm_panel.shape}")
    print(f"Columns          : {list(firm_panel.columns)}")
    print(f"Unique firms     : {firm_panel['permno'].nunique():,}")
    print(f"Date range       : {firm_panel['year_month'].min()}  ->  {firm_panel['year_month'].max()}")
    
    firm_panel.to_csv(OUT_FIRM, index=False)
    print(f"\nSaved -> {OUT_FIRM}")

In [ ]:
# ── Fallback: if characteristics file has NO cluster column ─────────
# In this case we need to merge cluster assignments from a separate source.
#
# Option A: If you have a file mapping (permno, year_month) -> cluster,
#           set ASSIGNMENT_FILE below and run this cell.
#
# Option B: Re-run your clustering code and save the firm-level assignments.

ASSIGNMENT_FILE = ""  # e.g. "/ssd1/.../firm_cluster_assignments.csv"

if "cluster" in chars.columns:
    print("Cluster column already present — this cell not needed.")

elif ASSIGNMENT_FILE and os.path.exists(ASSIGNMENT_FILE):
    print(f"Loading assignments from {ASSIGNMENT_FILE}...")
    assignments = pd.read_csv(ASSIGNMENT_FILE)
    assignments.columns = [c.lower() for c in assignments.columns]
    
    # Expect columns: permno, year_month, cluster
    assert "permno"     in assignments.columns, "Need 'permno' column"
    assert "year_month" in assignments.columns, "Need 'year_month' column"
    assert "cluster"    in assignments.columns, "Need 'cluster' column"
    
    firm_chars = chars.rename(columns={DATE_COL: "year_month", PERMNO_COL: "permno"})
    firm_chars = firm_chars.merge(
        assignments[["permno", "year_month", "cluster"]],
        on=["permno", "year_month"], how="inner"
    )
    print(f"After merge: {firm_chars.shape}")
    print("Now re-run Cells 7 and 8 to produce centroid_chars.csv and firm_panel.csv")

else:
    print("No cluster assignments available.")
    print()
    print("You need one of:")
    print("  A) A characteristics file that already has a 'cluster' column")
    print("  B) A separate firm-level cluster assignment file")
    print("     Set ASSIGNMENT_FILE above to its path.")
    print()
    print("The cluster_month_panel file only has AGGREGATE cluster stats,")
    print("not individual firm->cluster assignments.")